In [ ]:
import sys
from collections import defaultdict
from typing import List, Tuple, Dict, Optional
Board = List[int]

class TicTacToe:
    @staticmethod
    def winner(board: Board) -> int:
        lines = [(0,1,2),(3,4,5),(6,7,8),(0,3,6),(1,4,7),(2,5,8),(0,4,8),(2,4,6)]
        for a,b,c in lines:
            if board[a] == board[b] == board[c] != 0:
                return board[a]
        return 0

    @staticmethod
    def is_full(board: Board) -> bool:
        return 0 not in board

    @staticmethod
    def get_moves(board: Board) -> List[int]:
        return [i for i, cell in enumerate(board) if cell == 0]

    @staticmethod
    def make_move(board: Board, pos: int, player: int) -> Board:
        new_board = board[:]
        new_board[pos] = player
        return new_board

    @staticmethod
    def print_board(board: Board):
        sym = {1:'X', -1:'O', 0:' '}
        for i in range(0,9,3):
            print(' | '.join(sym[board[j]] for j in range(i,i+3)))
            if i < 6: print('-' * 5)

class LimitedMinimax:
    def __init__(self, max_table_size: int = 100):
        self.tt: Dict[Tuple[int,...], Tuple[int, int]] = {}  
        self.max_table_size = max_table_size
        self.nodes_visited = 0

    def clear_tt(self):
        self.tt.clear()

    def minimax(self, board: Board, depth: int, alpha: float, beta: float,
                maximizing: bool, orig_depth: int) -> int:
        self.nodes_visited += 1
        btuple = tuple(board)
        winner = TicTacToe.winner(board)
        if winner == 1: return (orig_depth - depth) / 10
        if winner == -1: return -(orig_depth - depth) / 10
        if TicTacToe.is_full(board): return 0.0

        tt_entry = self.tt.get(btuple)
        if tt_entry and tt_entry[1] >= depth:
            return tt_entry[0]

        best = -float('inf') if maximizing else float('inf')
        for move in TicTacToe.get_moves(board):
            new_board = TicTacToe.make_move(board, move, 1 if maximizing else -1)
            val = self.minimax(new_board, depth + 1, alpha, beta, not maximizing, orig_depth)
            if maximizing:
                best = max(best, val)
                alpha = max(alpha, best)
            else:
                best = min(best, val)
                beta = min(beta, best)
            if beta <= alpha:
                break
        if len(self.tt) >= self.max_table_size:
            self.tt.pop(next(iter(self.tt)))  
        self.tt[btuple] = (best, depth)

        return best

    def get_best_move(self, board: Board, max_time: float = 5.0, max_depth: int = 9) -> Optional[int]:
        self.clear_tt()
        self.nodes_visited = 0
        best_move = None
        best_score = -float('inf')

        for depth in range(1, max_depth + 1):
            moves = TicTacToe.get_moves(board)
            for move in moves:
                new_board = TicTacToe.make_move(board, move, 1)
                score = self.minimax(new_board, 1, -float('inf'), float('inf'), False, depth)
                if score > best_score:
                    best_score = score
                    best_move = move
            print(f"Depth {depth}: nodes={self.nodes_visited}, table_size={len(self.tt)}, move={best_move} score={best_score:.2f}")

        return best_move
class RandomAgent:
    def get_move(self, board: Board) -> int:
        import random
        return random.choice(TicTacToe.get_moves(board))

class UnlimitedMinimax(LimitedMinimax):
    def __init__(self):
        super().__init__(10**6)  

def play_game(player1, player2, start_player=1):
    board = [0] * 9
    current = start_player
    while True:
        TicTacToe.print_board(board)
        if current == 1:
            move = player1.get_best_move(board) if isinstance(player1, LimitedMinimax) else player1.get_move(board)
        else:
            move = player2.get_best_move(board) if isinstance(player2, LimitedMinimax) else player2.get_move(board)
        if move is None:
            return 0 
        board = TicTacToe.make_move(board, move, current)
        win = TicTacToe.winner(board)
        if win != 0 or TicTacToe.is_full(board):
            TicTacToe.print_board(board)
            return win

def experiment():
    limits = [10, 50, 100, 500, 1000]
    results = []
    unlimited = UnlimitedMinimax()
    limited_agents = {limit: LimitedMinimax(limit) for limit in limits}
    random_opp = RandomAgent()

    print("vs Random (basic opponent):")
    for limit, agent in list(limited_agents.items()) + [('Unlimited', unlimited)]:
        wins, draws, losses = 0, 0, 0
        for _ in range(50):
            res = play_game(agent, random_opp, 1)
            if res == 1: wins += 1
            elif res == 0: draws += 1
            else: losses += 1
        win_rate = wins / 50 * 100
        print(f"Memory {limit}: Win {win_rate:.1f}%, Draw {draws/50*100:.1f}%")
        results.append((limit, win_rate))

    print("\nvs Unlimited Minimax:")
    wins, draws, losses = 0, 0, 0
    for _ in range(50):
        res = play_game(unlimited, random_opp, -1)  
        if res == 1: wins += 1  
        elif res == 0: draws += 1
        else: losses += 1
    print(f"Unlimited vs Random: Win {wins/50*100:.1f}% (baseline)")

    limited100 = LimitedMinimax(100)
    wins_l, draws_l, losses_l = 0, 0, 0
    for _ in range(50):
        res = play_game(limited100, unlimited, 1)
        if res == 1: wins_l += 1
        elif res == 0: draws_l += 1
        else: losses_l += 1
    print(f"Limited(100) vs Unlimited: Win {wins_l/50*100:.1f}%, Draw {draws_l/50*100:.1f}%")

    return results

if __name__ == "__main__":
    if len(sys.argv) > 1 and sys.argv[1] == 'experiment':
        experiment()
    else:
        agent = LimitedMinimax(100)
        human = RandomAgent()  
        print("Limited agent (X) vs Random (O). Run 'experiment' for tests.")
        play_game(agent, human)


Limited agent (X) vs Random (O). Run 'experiment' for tests.
  |   |  
-----
  |   |  
-----
  |   |  
Depth 1: nodes=19149, table_size=100, move=4 score=0.00
Depth 2: nodes=38298, table_size=100, move=4 score=0.00
Depth 3: nodes=57447, table_size=100, move=4 score=0.00
Depth 4: nodes=76596, table_size=100, move=4 score=0.00
Depth 5: nodes=91333, table_size=100, move=4 score=0.00
Depth 6: nodes=107065, table_size=100, move=4 score=0.00
Depth 7: nodes=118767, table_size=100, move=4 score=0.00
Depth 8: nodes=137411, table_size=100, move=4 score=0.00
Depth 9: nodes=156413, table_size=100, move=4 score=0.00
  |   |  
-----
  | X |  
-----
  |   |  
Depth 1: nodes=3816, table_size=100, move=0 score=-0.20
Depth 2: nodes=7632, table_size=100, move=0 score=-0.10
Depth 3: nodes=11182, table_size=100, move=0 score=0.00
Depth 4: nodes=14868, table_size=100, move=0 score=0.00
Depth 5: nodes=17144, table_size=100, move=0 score=0.00
Depth 6: nodes=19604, table_size=100, move=0 score=0.10
Depth 7: no